# Datathon 2026 — Task 3: One-Click Sales Forecasting Pipeline

End-to-end reproducible pipeline. Run all cells (Cell → Run All or Restart & Run All) to:
1. Install dependencies
2. Verify data files
3. Run diagnostic EDA
4. Train the 4-layer ensemble → `submission_default.csv` (also copied as `submission_final.csv`)
5. Generate SHAP explainability plots

Total runtime on M-series Mac: ~3–5 minutes. The final submission is written to
`outputs/submissions/submission_final.csv` (Kaggle: 681,610).

## 1. Install dependencies

In [ ]:
%pip install -q --disable-pip-version-check \
    'pandas>=2.0' 'numpy<2.0' 'scikit-learn>=1.3' \
    'lightgbm>=4.0' 'xgboost>=2.0' 'catboost>=1.2' \
    'shap<0.46' 'optuna>=3.0' \
    'matplotlib>=3.5' 'scipy>=1.10'

## 2. Setup paths and imports

In [ ]:
import os, sys, shutil
from pathlib import Path

# Resolve project root (this notebook lives in notebooks/)
NB_PATH = Path.cwd()
if NB_PATH.name == 'notebooks':
    PROJECT_ROOT = NB_PATH.parent
else:
    PROJECT_ROOT = NB_PATH

SRC = PROJECT_ROOT / 'src'
DATA = PROJECT_ROOT / 'data'
OUT = PROJECT_ROOT / 'outputs'

sys.path.insert(0, str(SRC))
os.chdir(PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Source dir:   {SRC}')
print(f'Data dir:     {DATA}')
print(f'Output dir:   {OUT}')

## 3. Verify input data

The pipeline needs these files in `data/`:
- `sales.csv`, `sample_submission.csv`
- `customers.csv`, `geography.csv`, `inventory.csv`
- `order_items.csv`, `orders.csv`, `payments.csv`
- `products.csv`, `promotions.csv`, `returns.csv`
- `reviews.csv`, `shipments.csv`, `web_traffic.csv`

In [ ]:
import pandas as pd
REQUIRED = [
    'sales.csv', 'sample_submission.csv', 'customers.csv', 'geography.csv',
    'inventory.csv', 'order_items.csv', 'orders.csv', 'payments.csv',
    'products.csv', 'promotions.csv', 'returns.csv', 'reviews.csv',
    'shipments.csv', 'web_traffic.csv',
]
missing = [f for f in REQUIRED if not (DATA / f).exists()]
if missing:
    print('Missing files:')
    for m in missing:
        print(f'  - {DATA / m}')
    raise SystemExit('Place all required CSVs in data/ before continuing.')
print(f'All {len(REQUIRED)} input files present.')

sales = pd.read_csv(DATA / 'sales.csv', parse_dates=['Date'])
sub_template = pd.read_csv(DATA / 'sample_submission.csv', parse_dates=['Date'])
print(f"Train: {len(sales)} rows  ({sales.Date.min().date()} → {sales.Date.max().date()})")
print(f"Test:  {len(sub_template)} rows ({sub_template.Date.min().date()} → {sub_template.Date.max().date()})")

## 4. Diagnostic EDA

Yearly stats, day-of-week pattern, monthly seasonality, COGS/Revenue distribution.
Sanity-check that the data matches expectations.

In [ ]:
import analyze
analyze.main()

## 5. Train the 4-layer ensemble

**Architecture:**
- Layer 1: 3 GBM boosters (LightGBM + XGBoost + CatBoost), each with 1 base + 4 quarterly specialists
- Layer 2: Ridge on z-score normalized features
- Layer 3: Two-layer ensemble (Ridge + GBM blend)
- Layer 4: Auto-tuned scalar calibration on hold-out 2022-H2

Sample weighting emphasizes 2014–2018 (stable mid-period) by 100×.
Target transform: `log1p`. Output: `outputs/submissions/submission_default.csv` (Kaggle: 681,610).

In [ ]:
import train
train.main()

# Mark the default output as the final submission
src_path = OUT / 'submissions' / 'submission_default.csv'
dst_path = OUT / 'submissions' / 'submission_final.csv'
shutil.copy(src_path, dst_path)
print(f'\n[FINAL] Submission ready at: {dst_path}')

## 6. SHAP explainability

Generates feature-importance plots, SHAP summary beeswarms, dependence plots,
and a markdown report — required by the competition's explainability
constraint (Phần 3, Ràng buộc 3).

Output: `outputs/shap/SHAP_REPORT.md` and ~28 PNG plots.

In [ ]:
import explain
explain.main()

## 7. Submission preview

In [ ]:
final = pd.read_csv(OUT / 'submissions' / 'submission_final.csv')
print('FINAL submission summary:')
print(f'  Path: outputs/submissions/submission_final.csv')
print(f'  Rows: {len(final)}  Cols: {list(final.columns)}')
print(f'  Revenue mean: {final.Revenue.mean():,.0f}  '
      f'min: {final.Revenue.min():,.0f}  max: {final.Revenue.max():,.0f}')
print(f'  COGS mean:    {final.COGS.mean():,.0f}  '
      f'min: {final.COGS.min():,.0f}  max: {final.COGS.max():,.0f}')
print(f'  Avg margin:   {1 - (final.COGS / final.Revenue.clip(1)).mean():.3f}')
print()
print('First 5 rows:')
print(final.head().to_string(index=False))
print()
print('Last 5 rows:')
print(final.tail().to_string(index=False))

In [ ]:
# Quick visualization of the forecast trajectory
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

final['Date'] = pd.to_datetime(final['Date'])
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax0 = axes[0]
ax0.plot(sales.Date, sales.Revenue, lw=0.6, label='train', color='steelblue')
ax0.plot(final.Date, final.Revenue, lw=0.9, label='forecast', color='orange')
ax0.set_title('Revenue: history (2012–2022) + forecast (2023-01-01 → 2024-07-01)')
ax0.set_ylabel('Revenue (VND)')
ax0.legend(loc='upper left')

ax1 = axes[1]
ax1.plot(sales.Date, sales.COGS, lw=0.6, label='train', color='steelblue')
ax1.plot(final.Date, final.COGS, lw=0.9, label='forecast', color='orange')
ax1.set_title('COGS: history + forecast')
ax1.set_ylabel('COGS (VND)')
ax1.set_xlabel('Date')
ax1.legend(loc='upper left')

plt.tight_layout()
plt.show()

## ✅ Pipeline complete

Final submission file (upload this to Kaggle):

**`outputs/submissions/submission_final.csv`** — verified Kaggle public score **681,610**.

All intermediate logs, SHAP plots, and metadata are in `outputs/`.